In [0]:
# Demo dataset from Databricks
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/"))

In [0]:
# Copy data to a Unity Catalog volume
dbutils.fs.cp(
    "/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/",
    "/Volumes/catalog_s/default/data/nyctaxi_yellow/",
    recurse=True
)
display(
    dbutils.fs.ls(
        "/Volumes/catalog_s/default/data/nyctaxi_yellow/"
    )
)

In [0]:
df_parquet = spark.read.format(
    "delta"
).load(
    "/Volumes/catalog_s/default/data/nyctaxi_yellow/"
)
display(df_parquet)

In [0]:
# Check filter data
#select count(1) from nyctaxi where vendor_id = 'VTS' and trip_distance > 1.8

df_parquet.where("vendor_id = 'VTS' and trip_distance > 1.8").count()

In [0]:
display(
    df_parquet.select(
        "vendor_id"
    ).distinct()
)

In [0]:
df_parquet.write.format(
    "delta"
).mode(
    "overwrite"
).partitionBy(
    "vendor_id"
).saveAsTable(
    "catalog_s.default.nyctaxi_partitioned"
)

In [0]:
display(dbutils.fs.ls("/Volumes/catalog_s/default/data/"))

In [0]:
df_partitioned = spark.table(
    "catalog_s.default.nyctaxi_partitioned"
)

df_partitioned.where(
    "vendor_id = 'VTS' and trip_distance > 1.8"
).count()

In [0]:
%sql

select count(1) from catalog_s.default.nyctaxi_partitioned where vendor_id = 'VTS' and trip_distance > 1.8